<a href="https://colab.research.google.com/github/shivaniprasad22/BookRecommendationSystem/blob/Master/Book_Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/bookrecommendationsystem')

Mounted at /content/bookrecommendationsystem


In [ ]:
import numpy as np
import pandas as pd
import pickle
import os

import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import euclidean_distances
from scipy.spatial.distance import cdist

import warnings
warnings.filterwarnings("ignore")

In [ ]:
books = pd.read_csv('/content/bookrecommendationsystem/MyDrive/bookrecommendationsystem/books.csv')
users = pd.read_csv('/content/bookrecommendationsystem/MyDrive/bookrecommendationsystem/users.csv')
ratings = pd.read_csv('/content/bookrecommendationsystem/MyDrive/bookrecommendationsystem/ratings.csv')

In [ ]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [ ]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [ ]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271360, 8)
(1149780, 3)
(278858, 3)


In [ ]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [ ]:
users.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [ ]:
ratings.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [ ]:
books.duplicated().sum()

0

In [ ]:
ratings.duplicated().sum()

0

In [ ]:
users.duplicated().sum()

0

## Popularity Based Recommender System

In [ ]:
ratings_with_name = ratings.merge(books,on='ISBN')

In [ ]:
num_rating_df = ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating':'num_ratings'},inplace=True)
num_rating_df

,Book-Title,num_ratings
0,A Light in the Storm: The Civil War Diary of ...,4
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1
4,Beyond IBM: Leadership Marketing and Finance ...,1
...,...,...
241066,Ã?Â?lpiraten.,2
241067,Ã?Â?rger mit Produkt X. Roman.,4
241068,Ã?Â?sterlich leben.,1
241069,Ã?Â?stlich der Berge.,3


In [ ]:
# Convert Book-Rating to numeric, coercing errors to NaN
ratings_with_name['Book-Rating'] = pd.to_numeric(ratings_with_name['Book-Rating'], errors='coerce')

# Drop rows with NaN values in Book-Rating (optional, based on your requirements)
ratings_with_name = ratings_with_name.dropna(subset=['Book-Rating'])

# Calculate the average ratings
avg_rating_df = ratings_with_name.groupby('Book-Title')['Book-Rating'].mean().reset_index()

# Rename columns for clarity
avg_rating_df.columns = ['Book-Title', 'Average-Rating']

# Display the result
print(avg_rating_df)


                                               Book-Title  Average-Rating
0        A Light in the Storm: The Civil War Diary of ...        2.250000
1                                   Always Have Popsicles        0.000000
2                    Apple Magic (The Collector's series)        0.000000
3        Ask Lily (Young Women of Faith: Lily Series, ...        8.000000
4        Beyond IBM: Leadership Marketing and Finance ...        0.000000
...                                                   ...             ...
241066                                      Ã?Â?lpiraten.        0.000000
241067                     Ã?Â?rger mit Produkt X. Roman.        5.250000
241068                                Ã?Â?sterlich leben.        7.000000
241069                              Ã?Â?stlich der Berge.        2.666667
241070                                  Ã?Â?thique en toc        4.000000

[241071 rows x 2 columns]


In [ ]:
import pandas as pd

# Assuming ratings_with_name is your DataFrame

# Inspect unique values in Book-Rating column
print(ratings_with_name['Book-Rating'].unique())

# Convert Book-Rating to numeric, coercing errors to NaN
ratings_with_name['Book-Rating'] = pd.to_numeric(ratings_with_name['Book-Rating'], errors='coerce')

# Drop rows with NaN values in Book-Rating
ratings_with_name = ratings_with_name.dropna(subset=['Book-Rating'])

# Calculate the average ratings
avg_rating_df = ratings_with_name.groupby('Book-Title')['Book-Rating'].mean().reset_index()

# Rename columns for clarity
avg_rating_df.columns = ['Book-Title', 'Average-Rating']

# Display the result
print(avg_rating_df)


[ 0  5  9  8  6  7  4 10  3  2  1]
                                               Book-Title  Average-Rating
0        A Light in the Storm: The Civil War Diary of ...        2.250000
1                                   Always Have Popsicles        0.000000
2                    Apple Magic (The Collector's series)        0.000000
3        Ask Lily (Young Women of Faith: Lily Series, ...        8.000000
4        Beyond IBM: Leadership Marketing and Finance ...        0.000000
...                                                   ...             ...
241066                                      Ã?Â?lpiraten.        0.000000
241067                     Ã?Â?rger mit Produkt X. Roman.        5.250000
241068                                Ã?Â?sterlich leben.        7.000000
241069                              Ã?Â?stlich der Berge.        2.666667
241070                                  Ã?Â?thique en toc        4.000000

[241071 rows x 2 columns]


In [ ]:
popular_df = num_rating_df.merge(avg_rating_df,on='Book-Title')


In [ ]:
popular_df

,Book-Title,num_ratings,Average-Rating
0,A Light in the Storm: The Civil War Diary of ...,4,2.250000
1,Always Have Popsicles,1,0.000000
2,Apple Magic (The Collector's series),1,0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1,8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,1,0.000000
...,...,...,...
241066,Ã?Â?lpiraten.,2,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,4,5.250000
241068,Ã?Â?sterlich leben.,1,7.000000
241069,Ã?Â?stlich der Berge.,3,2.666667


In [ ]:
popular_df = popular_df[popular_df['num_ratings']>=250].sort_values('Average-Rating',ascending=False).head(60)

In [ ]:
popular_df = popular_df.merge(books,on='Book-Title').drop_duplicates('Book-Title')[['Book-Title','Book-Author','Image-URL-M','num_ratings','Average-Rating']]

## Collaborative Filtering Based Recommender System


In [ ]:
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 200
padhe_likhe_users = x[x].index

In [ ]:
filtered_rating = ratings_with_name[ratings_with_name['User-ID'].isin(padhe_likhe_users)]

In [ ]:
y = filtered_rating.groupby('Book-Title').count()['Book-Rating']>=50
famous_books = y[y].index

In [ ]:
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]

In [ ]:
pt = final_ratings.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating')

In [ ]:
pt.fillna(0,inplace=True)

In [ ]:
pt

User-ID,254,2276,2766,2977,3363,4017,4385,6251,6323,6543,...,271705,273979,274004,274061,274301,274308,275970,277427,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity_scores = cosine_similarity(pt)

In [ ]:
similarity_scores.shape

(706, 706)

In [ ]:
def recommend(book_name):
    # index fetch
    index = np.where(pt.index==book_name)[0][0]
    similar_items = sorted(list(enumerate(similarity_scores[index])),key=lambda x:x[1],reverse=True)[1:5]

    data = []
    for i in similar_items:
        item = []
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))

        data.append(item)

    return data

In [ ]:
recommend('1984')

[['Animal Farm',
  'George Orwell',
  'http://images.amazon.com/images/P/0451526341.01.MZZZZZZZ.jpg'],
 ["The Handmaid's Tale",
  'Margaret Atwood',
  'http://images.amazon.com/images/P/0449212602.01.MZZZZZZZ.jpg'],
 ['Brave New World',
  'Aldous Huxley',
  'http://images.amazon.com/images/P/0060809833.01.MZZZZZZZ.jpg'],
 ['The Vampire Lestat (Vampire Chronicles, Book II)',
  'ANNE RICE',
  'http://images.amazon.com/images/P/0345313860.01.MZZZZZZZ.jpg']]

In [ ]:
pt.index[545]

"The Handmaid's Tale"

In [ ]:
import pickle
pickle.dump(popular_df,open('popular.pkl','wb'))

In [ ]:
books.drop_duplicates('Book-Title')

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271354,0449906736,Flashpoints: Promise and Peril in a New World,Robin Wright,1993,Ballantine Books,http://images.amazon.com/images/P/0449906736.0...,http://images.amazon.com/images/P/0449906736.0...,http://images.amazon.com/images/P/0449906736.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


In [ ]:
import random

In [ ]:
def randomno():
    random_no =(random.randint(0,242135))
    rimage=books._get_value(random_no, 'Image-URL-M')
    rbook=books._get_value(random_no, 'Book-Title')
    rauthor=books._get_value(random_no, 'Book-Author')
    print(rimage,rbook,rauthor)
    return [rimage,rbook,rauthor]


In [ ]:
randomno()

http://images.amazon.com/images/P/0471197335.01.MZZZZZZZ.jpg Corporate Information Factory William H. Inmon


['http://images.amazon.com/images/P/0471197335.01.MZZZZZZZ.jpg',
 'Corporate Information Factory',
 'William H. Inmon']

In [ ]:
!pip install flask pyngrok

# Verify the installation
!ngrok version

# Install pyngrok
!pip install pyngrok

!ngrok authtoken 2i9I04T7kHgOg6G4TQmx98IhrGp_83cxGJ8fwU2C2NZBx3o5d

ngrok version 3.12.0
pyngrok version 7.1.6
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
pickle.dump(pt,open('pt.pkl','wb'))
pickle.dump(books,open('books.pkl','wb'))
pickle.dump(similarity_scores,open('similarity_scores.pkl','wb'))

In [ ]:
!pip install requests


In [ ]:
from flask import Flask, request, render_template
import pickle
import numpy as np
import random
import requests
from pyngrok import ngrok

# Load the necessary pickle files
popular_df = pickle.load(open('popular.pkl', 'rb'))
pt = pickle.load(open('pt.pkl', 'rb'))
books = pickle.load(open('books.pkl', 'rb'))
similarity_scores = pickle.load(open('similarity_scores.pkl', 'rb'))


template_folder = '/content/bookrecommendationsystem/MyDrive/bookrecommendationsystem/templates'
app = Flask(__name__, template_folder=template_folder)

# Function to get a high-quality image from Google Books API
def get_high_quality_image(title):
    google_books_api = "https://www.googleapis.com/books/v1/volumes"
    params = {
        'q': title,
        'maxResults': 1
    }
    try:
        response = requests.get(google_books_api, params=params)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()
        if 'items' in data:
            return data['items'][0]['volumeInfo'].get('imageLinks', {}).get('thumbnail', None)
    except (requests.RequestException, KeyError) as e:
        print(f"Error fetching image for {title}: {e}")
    return None

@app.route('/')
@app.route('/index')
def index():
    images = []
    for title in popular_df['Book-Title'].values:
        image = get_high_quality_image(title)
        if image is None:
            image = '/path/to/default/image.jpg'  # Use a default image if no high-quality image is found
        images.append(image)
    return render_template('index.html',
                           book_name=list(popular_df['Book-Title'].values),
                           author=list(popular_df['Book-Author'].values),
                           image=images,
                           votes=list(popular_df['num_ratings'].values),
                           rating=list(popular_df['Average-Rating'].values)  # Ensure this column exists
                           )

@app.route('/recommend')
def recommend_ui():
    return render_template('recommend.html')

@app.route('/recommend_books', methods=['POST'])
def recommend():
    user_input = request.form.get('user_input')
    try:
        index = np.where(pt.index == user_input)[0][0]
    except IndexError:
        return render_template('recommend.html', data=[['Book not found']])
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:5]

    data = []
    for i in similar_items:
        item = []
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        image = get_high_quality_image(pt.index[i[0]])
        if image is None:
            image = '/path/to/default/image.jpg'  # Use a default image if no high-quality image is found
        item.append(image)
        data.append(item)

    return render_template('recommend.html', data=data)

@app.route('/randomno', methods=["POST", "GET"])
def randomno():
    random_no = random.randint(0, len(books) - 1)
    rimage = books._get_value(random_no, 'Image-URL-M')
    rbook = books._get_value(random_no, 'Book-Title')
    rauthor = books._get_value(random_no, 'Book-Author')
    high_quality_image = get_high_quality_image(rbook)
    if high_quality_image:
        rimage = high_quality_image
    return render_template('randomno.html', img=rimage, book=rbook, auth=rauthor)

# Start Ngrok tunnel
ngrok_tunnel = ngrok.connect(5000)
print(f" * Ngrok tunnel: {ngrok_tunnel.public_url}")

# Run Flask app
app.run(port=5000, use_reloader=False)


 * Ngrok tunnel: https://b0d8-34-125-130-2.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [28/Jun/2024 14:21:20] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Jun/2024 14:21:21] "GET /path/to/default/image.jpg HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [28/Jun/2024 14:21:22] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [28/Jun/2024 14:21:39] "GET /recommend HTTP/1.1" 200 -
